# Evaluating LLM and RAG systems

Machine Learning Engineer track, lesson MLE10. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0. The text is the site's own: eight lessons of the SQL stage, also CC BY 4.0.

**Goals:** write a golden set and a trap set for the retrieval system of lesson MLE9, score every answer on three checks, see how far a language-model judge can be trusted, rerun the evaluation after a prompt change, and report the cost of an answer next to its quality.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. No key and no account are needed: both models are open and run in this notebook. A free CPU runtime is enough (the language model is about 3 GB to download, and each evaluation run takes a few minutes); a free GPU runtime is faster.

## Setup

In [ ]:
import importlib.metadata
import math
import numbers
import re
import statistics
import subprocess
import sys
import time
import urllib.request
from typing import Literal

for pin in ["transformers==5.24.1", "sentence-transformers==6.3.0", "pydantic==2.14.4"]:                                                         # the versions this notebook was run with
    package, version = pin.split("==")
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:                                                  # install the pinned version when Colab has another
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pin], check=True)

import numpy as np
import torch
from pydantic import BaseModel, ValidationError
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 2026       # the seed of the whole track
torch.set_num_threads(2)       # two threads, the size of a free Colab CPU

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## The corpus, the two models and the system under test

The corpus is the one of lesson MLE9: eight lessons of the SQL stage, from the release tag `v1.0.0`. The two models are the small encoder `all-MiniLM-L6-v2` and the open model `Qwen2.5-1.5B-Instruct`, each pinned to one revision, with the `chat` function of lesson MLE8. Run these cells as they are.

In [ ]:
# The corpus: eight published lessons of the SQL stage (CC BY 4.0), read from the release tag v1.0.0, or from this repository's own copy of the same files.
TEXT_BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v1.0.0/src/content/docs/sql/"
TEXT_LOCAL = "../../src/content/docs/sql/"      # the same files, when this notebook runs inside a clone of the repository
LESSONS = ["null-handling", "group-by-having", "inner-join", "left-join-anti-join", "window-functions", "ctes", "dates-and-time", "case-expressions"]
SKIP_SECTIONS = {"Warm-up", "Finish it", "On your own", "Check yourself"}      # recall prompts and exercises are not teaching text


def read_lesson(slug):
    """The raw text of one lesson file: downloaded from the tag, or the local copy."""
    try:
        with urllib.request.urlopen(TEXT_BASE + slug + ".mdx", timeout=30) as reply:
            return reply.read().decode("utf-8")
    except Exception:
        with open(TEXT_LOCAL + slug + ".mdx", encoding="utf-8") as handle:
            return handle.read()


def paragraphs(raw):
    """The prose paragraphs of a lesson as (section heading, text): no front matter, figures, code, tables or exercise sections."""
    raw = re.sub(r"\A---\n.*?\n---\n", "", raw, flags=re.S)
    raw = re.sub(r"<figure.*?</figure>|<svg.*?</svg>|<RunSQL.*?/>|```.*?```", "", raw, flags=re.S)
    found, section = [], ""
    for block in re.split(r"\n\s*\n", raw):
        block = block.strip()
        if block.startswith("#"):
            section = block.lstrip("#").strip()
        elif block and section not in SKIP_SECTIONS and not block.startswith(("<", "|", "{/*")):
            block = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", re.sub(r"<[^>]+>", "", block)).replace("*", "").replace("`", "")
            block = re.sub(r"\s+", " ", block).strip()
            if len(block.split()) >= 8:                                           # a few words are a label, not a paragraph
                found.append((section, block))
    return found


CORPUS = {slug: paragraphs(read_lesson(slug)) for slug in LESSONS}
paras = [(slug, text) for slug in LESSONS for _, text in CORPUS[slug]]            # (lesson, paragraph) for every paragraph
texts, labels = [text for _, text in paras], [slug for slug, _ in paras]
print(len(paras), "paragraphs,", sum(len(text.split()) for text in texts), "words,", len(LESSONS), "lessons")

In [ ]:
EMB_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMB_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"                    # the exact version of the model files
emb = SentenceTransformer(EMB_NAME, revision=EMB_REVISION, device="cpu")
print(sum(p.numel() for p in emb.parameters()), emb.max_seq_length)

In [ ]:
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"                    # the exact version of the model files
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(LLM_NAME, revision=LLM_REVISION)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, revision=LLM_REVISION, dtype=torch.float32).to(DEVICE).eval()
END_MARKS = set(tok.convert_tokens_to_ids(["<|im_end|>", "<|endoftext|>"]))


def chat(messages, max_new_tokens=100, temperature=0.0, seed=None):
    """One request. temperature 0 picks the most likely token every time; above 0 it samples."""
    prompt = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"].to(DEVICE)
    settings = {"do_sample": False, "repetition_penalty": 1.0}
    if temperature > 0:
        settings = {"do_sample": True, "temperature": temperature, "top_p": 1.0, "top_k": 0, "repetition_penalty": 1.0}
        torch.manual_seed(SEED if seed is None else seed)
    start = time.perf_counter()
    with torch.no_grad():
        out = llm.generate(input_ids=prompt, attention_mask=torch.ones_like(prompt), max_new_tokens=max_new_tokens, **settings)
    seconds = time.perf_counter() - start
    written = out[0][prompt.shape[1]:].tolist()
    return {"text": tok.decode(written, skip_special_tokens=True), "prompt_tokens": prompt.shape[1],
            "completion_tokens": sum(1 for t in written if t not in END_MARKS), "seconds": seconds}


print(sum(p.numel() for p in llm.parameters()), DEVICE)

The system of lesson MLE9, in one cell, and its settings: chunks of 120 words, the 3 best chunks as numbered sources, a JSON answer with the numbers of the sources it used, and a refusal when even the best chunk has a cosine below `THRESHOLD`. `PROMPTS["v1"]` is the system prompt of lesson MLE9. Run this cell as it is.

In [ ]:
def make_chunks(target_words):
    """Chunks of whole paragraphs, each (lesson, section, text), at most `target_words` words unless one paragraph is longer."""
    chunks = []
    for slug in LESSONS:
        current, count, head = [], 0, ""
        for section, text in CORPUS[slug]:
            size = len(text.split())
            if current and count + size > target_words:
                chunks.append((slug, head, " ".join(current)))
                current, count = [], 0
            if not current:
                head = section
            current.append(text)
            count += size
        if current:
            chunks.append((slug, head, " ".join(current)))
    return chunks


CHUNKS = make_chunks(120)
VECTORS = emb.encode([chunk[2] for chunk in CHUNKS])
THRESHOLD = 0.40                                                      # the cosine below which the system refuses without calling the model
PROMPTS = {"v1": ("Answer the question using only the numbered sources. Reply with JSON only, like {\"answer\": \"one or two sentences\", \"sources\": [1]}, "
                  "where sources lists the numbers of the sources you used. If the sources do not contain the answer, reply {\"answer\": null, \"sources\": []}.")}


class Cited(BaseModel):
    answer: str | None
    sources: list[int]


def search(question, k=3):
    """The k chunks most similar to the question, as (cosine, number of the chunk)."""
    scores = VECTORS @ emb.encode([question])[0]
    return [(round(float(scores[i]), 3), int(i)) for i in np.argsort(-scores)[:k]]


def answer(question, version="v1", k=3):
    """The whole system: search, refuse when nothing is close, otherwise ask the model and check its reply."""
    found = search(question, k)
    if found[0][0] < THRESHOLD:
        return {"found": found, "answer": None, "sources": [], "valid": True, "abstained": True, "prompt_tokens": 0, "completion_tokens": 0, "seconds": 0.0}
    sources = "\n".join(f"[{n}] {CHUNKS[i][2]}" for n, (score, i) in enumerate(found, start=1))
    reply = chat([{"role": "system", "content": PROMPTS[version]}, {"role": "user", "content": f"Sources:\n{sources}\n\nQuestion: {question}"}], max_new_tokens=100)
    try:
        parsed = Cited.model_validate_json(reply["text"])
        valid = all(1 <= n <= k for n in parsed.sources)
        return {**reply, "found": found, "answer": parsed.answer, "sources": parsed.sources, "valid": valid, "abstained": False}
    except ValidationError:
        return {**reply, "found": found, "answer": None, "sources": [], "valid": False, "abstained": False}


print(len(CHUNKS), THRESHOLD)

## The golden set and the trap set

`GOLD` is 16 questions written for this lesson after the 16 of lesson MLE9, two for each lesson, and never used to choose anything. Each holds the question, the lesson that answers it, phrases of the paragraph that holds the answer and patterns that a correct answer must match. `TRAPS` are 6 questions that the eight lessons do not answer: three that use SQL words and three that do not. Run this cell as it is.

In [ ]:
GOLD = [
    ('Why does a NOT IN list that holds a NULL return no rows?', 'null-handling', ['When the list holds a NULL, a value that is not in it gets NULL, not true'], ['NULL', 'not true|unknown|no rows|nothing|none']),
    ('Do aggregate functions such as AVG count missing values?', 'null-handling', ['Most aggregate functions skip NULL', 'Most SQL aggregate functions ignore NULL'], ["skip|ignore|not count|do not count|don't count"]),
    ('In which order does SQL work through the clauses of a query?', 'group-by-having', ['SQL works through the clauses in this order: FROM, WHERE, GROUP BY'], ['(?s)FROM.*WHERE.*GROUP BY']),
    ('What does every group become after GROUP BY?', 'group-by-having', ['every group becomes one row', 'turns every group into a single row'], ['(one|single) row']),
    ('What does the ON condition of a join name?', 'inner-join', ['ON names the key on each side'], ['key']),
    ('Is JOIN on its own the same as INNER JOIN?', 'inner-join', ['JOIN alone means the same as INNER JOIN'], ['same|yes']),
    ("What goes in the right table's columns when a row of a LEFT JOIN finds no match?", 'left-join-anti-join', ['with NULL (an empty value, S4) in every column of the right table'], ['NULL']),
    ('What does a RIGHT JOIN keep that an inner join drops?', 'left-join-anti-join', ["RIGHT keeps the right table's unpaired rows"], ['right']),
    ('What does PARTITION BY do inside OVER?', 'window-functions', ['it fences the rows into blocks', 'PARTITION BY picks the block'], ['block|fence|split|divide|group|partition']),
    ('Which rows does the default frame cover when OVER has an ORDER BY?', 'window-functions', ["With an ORDER BY and nothing else, the frame runs from the block's first row to the current row", 'from the first row to the current row, ties included'], ['(?s)first row.*current row']),
    ('How do you add a second step after the first step of a CTE?', 'ctes', ['After a comma, the next step is defined the same way', 'commas separate the steps'], ['comma']),
    ('How do you debug one step of a chain of CTEs?', 'ctes', ['point the last SELECT at one step and run it'], ['last SELECT|one step']),
    ('Does date_trunc round to the nearest month?', 'dates-and-time', ['It cuts down, never up'], ['down|never up|not round|\\bno\\b']),
    ('What does the isodow function return?', 'dates-and-time', ['isodow (the weekday number, Monday = 1 to Sunday = 7)'], ['weekday|Monday']),
    ('Which test should come first in a CASE when two tests overlap?', 'case-expressions', ['write the strictest test first'], ['strict']),
    ('What does the ELSE branch of a CASE give?', 'case-expressions', ['An optional ELSE gives the value for a row that none of the tests accepts'], ["none of the tests|no test|no WHEN|otherwise|any other|every other|not match|doesn't match|does not match"]),
]
TRAPS = [
    ('How do I delete rows from a table?', 'near'),
    ('What does INSERT INTO do?', 'near'),
    ('How do I write a recursive query?', 'near'),
    ('How do I plot a histogram in Python?', 'far'),
    ('What is overfitting?', 'far'),
    ('What is the tallest mountain in Europe?', 'far'),
]
print(len(GOLD), "golden questions,", len(TRAPS), "traps")

## Unit 1: A golden set and three checks

**Predict first.** The system answers each of the 16 golden questions. Each answer gets three yes-or-no checks: was a chunk with the phrase of the answer among the 3 retrieved, does the answer match the patterns, and does a source the answer cites hold the phrase? Which check fails most often, and how many answers pass all three?

In [ ]:
def check_one(item, result):
    """The three checks of one golden question."""
    _, _, phrases, patterns = item
    retrieved = any(phrase in CHUNKS[i][2] for _, i in result["found"] for phrase in phrases)       # a chunk with the answer was found
    matches = result["answer"] is not None and all(re.search(pattern, result["answer"], re.I) for pattern in patterns)
    cites = result["valid"] and any(phrase in CHUNKS[result["found"][n - 1][1]][2] for n in result["sources"] for phrase in phrases)   # a cited source holds it
    return retrieved, matches, cites


runs = {"v1": {"gold": [answer(item[0], "v1") for item in GOLD]}}
checks = [check_one(item, result) for item, result in zip(GOLD, runs["v1"]["gold"])]
print([sum(row[k] for row in checks) for k in range(3)], sum(all(row) for row in checks))

## Unit 2: Traps and refusals

**Predict first.** The 6 trap questions have no answer in the corpus, so the right reply to each is a refusal. How many will the system refuse, how many of those does the threshold stop before the model is called, and how many of the 16 golden questions were refused when they should have been answered?

In [ ]:
runs["v1"]["traps"] = [answer(question, "v1") for question, _ in TRAPS]
for (question, kind), result in zip(TRAPS, runs["v1"]["traps"]):
    print(kind, result["found"][0][0], "refused" if result["answer"] is None else "answered", "by the threshold" if result["abstained"] else "", "|", question)
refused_traps = sum(result["answer"] is None for result in runs["v1"]["traps"])
refused_gold = sum(result["answer"] is None for result in runs["v1"]["gold"])
print(refused_traps, sum(result["abstained"] for result in runs["v1"]["traps"]), refused_gold)
for item, result in zip(GOLD, runs["v1"]["gold"]):
    if result["answer"] is None:
        print(result["found"][0][0], item[0])

## Unit 3: A language model as the judge

**Predict first.** The same model that wrote the answers is asked to judge them against a reference, with the rubric that follows. The judge sees 16 answers that are wrong by construction (each is the answer to the next question) and the 16 real answers. How many wrong answers will it accept, and how often will its verdict on the real answers agree with the patterns?

In [ ]:
JUDGE = ("You grade an answer to a question about SQL. You get the question, a reference sentence from the lesson, and a candidate answer. "
         "Reply with exactly one word: yes if the candidate answer says what the reference says, no if it does not.")


def judge(question, reference, candidate):
    """True when the model says the candidate agrees with the reference."""
    prompt = f"Question: {question}\nReference: {reference}\nCandidate answer: {candidate}"
    return chat([{"role": "system", "content": JUDGE}, {"role": "user", "content": prompt}], max_new_tokens=3)["text"].strip().lower().startswith("yes")


gold_answers = [result["answer"] or "no answer" for result in runs["v1"]["gold"]]
wrong_answers = gold_answers[1:] + gold_answers[:1]                          # each question gets the answer of the next one
accepted_wrong = [judge(item[0], item[2][0], wrong) for item, wrong in zip(GOLD, wrong_answers)]
verdicts = [judge(item[0], item[2][0], real) for item, real in zip(GOLD, gold_answers)]
answered = [i for i, result in enumerate(runs["v1"]["gold"]) if result["answer"] is not None]       # the refusals have no answer to judge
agree = sum(verdicts[i] == checks[i][1] for i in answered)
print(sum(accepted_wrong), sum(verdicts[i] for i in answered), agree, len(answered))

## Unit 4: Run it again after a change, and read the cost

**Predict first.** Version v2 of the prompt adds one sentence: answer in at most 15 words. The whole evaluation is run again with it. Will v2 cost less per answer, and will it keep every golden question that v1 answered correctly? Prices are the assumed ones of lesson MLE8: $1.00 per million prompt tokens and $5.00 per million completion tokens.

In [ ]:
PROMPTS["v2"] = PROMPTS["v1"] + " Answer in at most 15 words."                  # the change under test
runs["v2"] = {"gold": [answer(item[0], "v2") for item in GOLD], "traps": [answer(question, "v2") for question, _ in TRAPS]}
PRICE_IN, PRICE_OUT = 1.00, 5.00                                               # dollars per million tokens: assumed, not any provider's price


def report(version):
    """The numbers of one run: passes, refusals of traps, tokens, dollars per 1,000 answers (answers that called the model), median seconds."""
    gold, traps = runs[version]["gold"], runs[version]["traps"]
    passes = [all(check_one(item, result)) for item, result in zip(GOLD, gold)]
    called = [result for result in gold + traps if not result["abstained"]]
    cost = (statistics.mean(r["prompt_tokens"] for r in called) * PRICE_IN + statistics.mean(r["completion_tokens"] for r in called) * PRICE_OUT) / 1_000_000 * 1000
    return passes, sum(result["answer"] is None for result in traps), round(statistics.mean(r["prompt_tokens"] for r in called)), round(statistics.mean(r["completion_tokens"] for r in called)), round(cost, 3), round(statistics.median(r["seconds"] for r in called), 1)


for version in ["v1", "v2"]:
    passes, refused, prompt_mean, completion_mean, cost, seconds = report(version)
    print(version, sum(passes), refused, prompt_mean, completion_mean, cost, seconds)
before, after = report("v1")[0], report("v2")[0]
print("lost", [i for i in range(len(GOLD)) if before[i] and not after[i]], "gained", [i for i in range(len(GOLD)) if after[i] and not before[i]])

## Finish it

### Exercise 1: Finish it

**Task:** **A fourth check.** Add a check for length to the three of unit 1: an answer is short when it has at most 20 words. Put two numbers in the list `answer_1`: how many of the 16 golden answers of `runs["v1"]` are short, and how many of the answers that pass all three checks are short.

**Expected:** `answer_1` is a list of two whole numbers.

In [ ]:
gold_v1 = runs["v1"]["gold"]
passed = [all(check_one(item, result)) for item, result in zip(GOLD, gold_v1)]                 # given: the answers that pass all three checks
short = [result["answer"] is not None and len(result["answer"].split()) <= 20 for result in gold_v1]
answer_1 = None  # replace None with [number of short answers, number of short answers among those that pass]

In [ ]:
_ = check("Exercise 1", "answer_1", [11, 6])

<details>
<summary>Hint</summary>

`passed` and `short` have one entry for each golden question, so `zip` pairs them.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_1 = [sum(short), sum(p and s for p, s in zip(passed, short))]
```

</details>

<details>
<summary>Solution</summary>

```python
gold_v1 = runs["v1"]["gold"]
passed = [all(check_one(item, result)) for item, result in zip(GOLD, gold_v1)]
short = [result["answer"] is not None and len(result["answer"].split()) <= 20 for result in gold_v1]
answer_1 = [sum(short), sum(p and s for p, s in zip(passed, short))]
```

[11, 6]: 11 of the 12 answers that the system gave are short (the four refusals have no answer to count), and 6 of the 7 answers that pass all three checks are short. A length check is cheap and says nothing about whether an answer is right.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Lesson by lesson.** Before you start, say in one sentence what you expect: a total over 16 questions can hide a lesson where the system never passes. Then write the cell yourself, in the empty cell below: for each of the eight lessons, in the order of `LESSONS`, count how many of its two golden questions pass all three checks in `runs["v1"]`, and put the eight counts in the list `answer_2`.

**Expected:** `answer_2` is a list of eight numbers, each 0, 1 or 2.

In [ ]:
_ = check("Exercise 2", "answer_2", [1, 2, 1, 1, 1, 0, 1, 0])

<details>
<summary>Hint</summary>

The lesson of a golden question is `item[1]`. Build the list of passes with `all(check_one(item, result))` and sum it for each lesson.

</details>

<details>
<summary>Half-solution</summary>

```python
passes = [all(check_one(item, result)) for item, result in zip(GOLD, runs["v1"]["gold"])]
answer_2 = [... for lesson in LESSONS]
```

</details>

<details>
<summary>Solution</summary>

```python
passes = [all(check_one(item, result)) for item, result in zip(GOLD, runs["v1"]["gold"])]
answer_2 = [sum(p for p, item in zip(passes, GOLD) if item[1] == lesson) for lesson in LESSONS]
```

[1, 2, 1, 1, 1, 0, 1, 0]: the group-by-having lesson passes both its questions, the `ctes` and `case-expressions` lessons pass none, and the others pass one. The total of 7 hides the two lessons where the system never passes.

</details>

## Check yourself

### Exercise 3: Check yourself

**Task:** **Spot the flaw.** A colleague says that the judge of unit 3 is reliable, because it accepted none of the 16 answers that are wrong by construction. Test the other side: give the judge each question with the first phrase of `GOLD[i][2]` as the candidate answer, an answer that is right by construction, and put the number of the 16 it accepts in `answer_3`.

**Expected:** `answer_3` is a whole number from 0 to 16.

In [ ]:
answer_3 = sum(accepted_wrong)                                          # colleague: "it accepted none of the wrong answers"
# replace this with the number of right-by-construction answers that the judge accepts

In [ ]:
_ = check("Exercise 3", "answer_3", 16, traps={0: "That is the wrong side of the test only: a judge that says no to everything also scores 0 there."})

<details>
<summary>Hint</summary>

`judge(question, reference, candidate)` takes the three texts; the reference and the candidate can be the same phrase.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_3 = sum(judge(item[0], item[2][0], ...) for item in GOLD)
```

</details>

<details>
<summary>Solution</summary>

```python
answer_3 = sum(judge(item[0], item[2][0], item[2][0]) for item in GOLD)
```

The answer is 16. The judge accepts every right-by-construction answer and rejected every wrong-by-construction one, so on the two extremes it behaves. The claim looked at one side only, where a judge that says no to everything would also score 0. The answers that matter sit between the extremes: on the 12 real answers of unit 3, the verdicts disagreed with the patterns twice.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")